# 📊 Proyecto: Análisis Comercial e Inmobiliario (2023–2024)
**Empresa:** Andes Capital Real Estate  
**Autor:** William  

## 📌 Contexto del Negocio
El sector inmobiliario requiere evaluar su desempeño comercial para comprender su crecimiento, rentabilidad y comportamiento de clientes. Este proyecto abarca la limpieza de datos, modelado dimensional en esquema estrella, creación de medidas DAX y la construcción de un resumen ejecutivo enfocado en la toma de decisiones estratégicas.

---

## 📂 Descripción de los Datasets
* `hecho_ventas_propiedades`: Tabla de hechos con transacciones de venta (`precio`, `cliente`, `propiedad`, `canal`, `fecha`, `comisión`).
* `dim_clientes`: Información demográfica y segmentación de clientes (`Primera vez`, `Inversionista`, `Alto patrimonio`).
* `dim_propiedades`: Características físicas y ubicación de las propiedades (`Casa`, `Departamento`, `Comercial`).
* `Dim_Fecha`: Tabla calendario dinámica creada para el análisis temporal.

---

## 🎯 Preguntas Principales de Negocio
1. **Desempeño General:** ¿Cuál es el ingreso total, volumen de ventas, ticket promedio y comisión generada?
2. **Análisis Comercial:** ¿Qué tipo de propiedad, segmento de cliente y canal de venta generan mayor valor?
3. **Análisis Temporal:** ¿Cómo evolucionan las ventas mes a mes y cuál es el crecimiento interanual (YoY)?
4. **Retención / Cohortes:** ¿Los clientes vuelven a comprar después de su primera transacción?

## 🧹 Paso 1: Limpieza y Auditoría de Datos

En este apartado se valida la integridad de los datos antes del modelado:
1. **Tipado de Datos:** Asegurar formato `Date` en fechas y numérico estricto en montos y comisiones.
2. **Valores Nulos:** Verificación de ausencia de nulos en llaves primarias y campos de montos.
3. **Duplicados:** Validación de unicidad de `id_cliente` en `dim_clientes` e `id_propiedad` en `dim_propiedades`.

In [4]:
import pandas as pd
import numpy as np

# Carga de datasets
ventas = pd.read_csv('hecho_ventas_propiedades.csv')
clientes = pd.read_csv('dim_clientes.csv')
propiedades = pd.read_csv('dim_propiedades.csv')

# 1. Transformación de tipos de datos
ventas['fecha_venta'] = pd.to_datetime(ventas['fecha_venta'])
ventas['precio_venta'] = pd.to_numeric(ventas['precio_venta'], errors='coerce')
ventas['monto_comision'] = pd.to_numeric(ventas['monto_comision'], errors='coerce')

# 2. Auditoría de Duplicados en Claves Primarias
dup_clientes = clientes['id_cliente'].duplicated().sum()
dup_propiedades = propiedades['id_propiedad'].duplicated().sum()

# 3. Auditoría de Nulos
nulos_ventas = ventas[['precio_venta', 'fecha_venta', 'id_cliente', 'id_propiedad']].isnull().sum()

print("=== INFORME DE AUDITORÍA DE DATOS ===")
print(f"✓ Duplicados en dim_clientes (id_cliente): {dup_clientes}")
print(f"✓ Duplicados en dim_propiedades (id_propiedad): {dup_propiedades}")
print("\n✓ Reporte de valores nulos en hechos:")
print(nulos_ventas)
print("\n✓ Tipos de datos confirmados correctamente.")

=== INFORME DE AUDITORÍA DE DATOS ===
✓ Duplicados en dim_clientes (id_cliente): 0
✓ Duplicados en dim_propiedades (id_propiedad): 0

✓ Reporte de valores nulos en hechos:
precio_venta    0
fecha_venta     0
id_cliente      0
id_propiedad    0
dtype: int64

✓ Tipos de datos confirmados correctamente.


## 📅 Paso 2: Dimensión Temporal (Dim_Fecha)

Para garantizar un correcto análisis temporal (YoY, YTD, MTD) y evitar desorden alfabético en los meses, se crea la tabla `Dim_Fecha` conectada dinámicamente al rango de transacciones.

// Código DAX ejecutado en Power BI para la creación de Dim_Fecha//

Dim_Fecha = 

ADDCOLUMNS(

    CALENDAR( MIN(hecho_ventas_propiedades[fecha_venta]), MAX(hecho_ventas_propiedades[fecha_venta] )),
    
    "Año", YEAR([Date]),
    
    "Mes", FORMAT([Date], "MMMM"),
    
    "Mes Numero", MONTH([Date]),
    
    "Año-Mes", FORMAT([Date], "YYYY-MM"))

## 🧩 Paso 3: Modelado de Datos (Esquema Estrella)

El modelo de datos se estructura en un **Esquema Estrella (Star Schema)** centrado en la tabla de hechos `hecho_ventas_propiedades`, garantizando relaciones de 1 a Muchos (1:*) con filtro en dirección única (*Single*):

* `dim_clientes[id_cliente]` (1) ───> (*) `hecho_ventas_propiedades[id_cliente]`
* `dim_propiedades[id_propiedad]` (1) ───> (*) `hecho_ventas_propiedades[id_propiedad]`
* `Dim_Fecha[Date]` (1) ───> (*) `hecho_ventas_propiedades[fecha_venta]`

## 📊 Paso 4: Documentación de Medidas DAX y Columnas Calculadas

A continuación se detallan las medidas implementadas para los KPIs principales, análisis de participación, inteligencia de tiempo y cálculos requeridos para la matriz de cohortes.

-- =========================================================

-- 1. MEDIDAS BASE -- 

=========================================================

Ingreso Total = SUM(hecho_ventas_propiedades[precio_venta])


Cantidad de Ventas = COUNTROWS(hecho_ventas_propiedades)


Ticket Promedio = DIVIDE([Ingreso Total], [Cantidad de Ventas])


Comisión Total = SUM(hecho_ventas_propiedades[monto_comision])


-- =========================================================

-- 2. MEDIDAS DE CONTEXTO DE FILTRO (% PARTICIPACIÓN)

-- =========================================================

% Participación Tipo Propiedad = 
DIVIDE(
    [Ingreso Total], 
    CALCULATE([Ingreso Total], ALL(hecho_ventas_propiedades[tipo_propiedad])))
    

% Participación Canal Venta = 
DIVIDE(
    [Ingreso Total], 
    CALCULATE([Ingreso Total], ALL(hecho_ventas_propiedades[canal_venta])))
    

-- =========================================================

-- 3. INTELIGENCIA DE TIEMPO

-- =========================================================

Ingreso Año Anterior = CALCULATE([Ingreso Total], SAMEPERIODLASTYEAR(Dim_Fecha[Date]))


% Crecimiento YoY = DIVIDE([Ingreso Total] - [Ingreso Año Anterior], [Ingreso Año Anterior])


Ingreso YTD = TOTALYTD([Ingreso Total], Dim_Fecha[Date])


-- =========================================================

-- 4. COLUMNAS CALCULADAS PARA COHORTES (En hecho_ventas_propiedades)

-- =========================================================

Fecha Primera Compra Cliente = 
CALCULATE(
    MIN(hecho_ventas_propiedades[fecha_venta]), 
    ALLEXCEPT(hecho_ventas_propiedades, hecho_ventas_propiedades[id_cliente]))
    

Mes Cohorte = FORMAT(hecho_ventas_propiedades[Fecha Primera Compra], "YYYY-MM")


Mes Venta = FORMAT(hecho_ventas_propiedades[fecha_venta], "YYYY-MM")


-- =========================================================

-- 5. TÍTULO DINÁMICO (Para página Detalle Cliente)

-- =========================================================

Titulo Cliente Seleccionado = 
VAR ClienteSel = SELECTEDVALUE(dim_clientes[id_cliente], "Todos los Clientes")
VAR SegmentoSel = SELECTEDVALUE(dim_clientes[segmento_comprador], "Todos los Segmentos")
RETURN
"Evolución de Ingresos — " & ClienteSel & " (" & SegmentoSel & ")"

## 🖼️ Paso 5: Estructura del Reporte

El dashboard interactivo consta de 4 páginas principales:
1. **Overview Ejecutivo:** Visión general de la empresa, KPIs, tendencia temporal y mapa comparativo por ciudad.
2. **Análisis Comercial:** Desglose de ingresos por tipo de propiedad, canal de venta, segmento de comprador y tabla condicional de combinaciones.
3. **Análisis de Cohortes:** Matriz de comportamiento de recompra en el tiempo.
4. **Detalle por Cliente:** Ranking individual, historial transaccional y título dinámico adaptado a los filtros activos.

---

### 💡 Guía de Lectura para la Matriz de Cohortes
* **Filas (Mes Cohorte):** Representa el mes en el que el grupo de clientes realizó su primera transacción.
* **Columnas (Mes Venta):** Representa el mes en que ocurren compras posteriores.
* **Diagonal Principal (1.00 / 100%):** Muestra el 100% de la cohorte adquirida en el mes base.
* **Valores Posteriores (Ej. 0.06, 0.16, 0.27):** Indican la proporción de clientes de esa cohorte que volvieron a comprar en ese mes específico. *Ejemplo: Un valor de 0.16 en la columna 2023-03 indica que el 16% de los clientes adquiridos en 2023-01 volvieron a realizar una compra en marzo de 2023.*

## 📝 Paso 6: Resumen Ejecutivo

### 📊 Métricas Principales

* **Ingreso Total:** $6,013 mil millones

* **Cantidad de Ventas:** 8,500 operaciones

* **Ticket Promedio:** $707.35 mil

* **Comisión Total:** $200.63 millones

---

### 🔍 Hallazgos Clave
1. **Liderazgo por Producto:** Las propiedades tipo **Casa** constituyen la principal fuente de facturación del negocio.
2. **Predominio del Canal:** El canal **Corredor** concentra casi 3 veces más transacciones e ingresos que la venta directa.
3. **Perfil del Comprador:** Los compradores del segmento **Primera vez** representan el mayor volumen de ingresos aportados.
4. **Comportamiento Geográfico:** **Ciudad de México** supera a Bogotá en un 17% en ingresos generados.
5. **Estacionalidad:** El crecimiento interanual (YoY) se mantiene positivo con picos marcados en marzo, abril, septiembre, octubre y noviembre.
6. **Retención de Clientes:** La matriz de cohortes refleja una caída severa de la recurrencia tras la primera transacción.

---

### 💡 Insights Strategics
* **Insight 1:** La alta dependencia en compradores de "Primera vez" indica una excelente capacidad de captación, pero una baja madurez en estrategias de fidelización y *Customer Lifetime Value* (CLV).
* **Insight 2:** La red de corredores externos es el motor comercial crítico de la empresa.
* **Insight 3:** El comportamiento de ventas es predecible en el año, permitiendo anticipar presupuestos de marketing antes de los meses de alta demanda.

---

### 🚀 Recomendaciones de Negocio
1. **Estrategia de Fidelización:** Diseñar programas de seguimiento postventa para incentivar la compra recurrente durante los primeros 3 a 6 meses posteriores a la primera transacción.
2. **Apalancamiento de Redes:** Fortalecer las comisiones e incentivos para el canal **Corredor**, dada su relevancia en los ingresos totales.
3. **Planificación Estacional:** Ejecutar campañas intensivas en febrero y agosto para maximizar los picos comerciales de marzo-abril y septiembre-noviembre.
4. **Optimización de Oferta:** Priorizar la captura de inventario para propiedades tipo **Casa** en Ciudad de México.